In [ ]:
!pip install --upgrade --quiet langchain langchain-community langchain-groq neo4j
#

In [ ]:
import langchain
print(f"Langchain version: {langchain.__version__}")

In [ ]:
# Graph DB configuraiton
NEO4J_URI="neo4j+s://xxxxx.databases.neo4j.io"
NEO4J_USERNAME="xxxxx"
NEO4J_PASSWORD="xxxxx"
NEO4J_DATABASE="xxxx"
AURA_INSTANCEID="xxxxx"
AURA_INSTANCENAME="Free instance"

In [ ]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [ ]:
from langchain_community.graphs import Neo4jGraph
graph=Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE)

In [ ]:
graph

In [ ]:
groq_api = "gsk_..."

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(temperature=0, groq_api_key=groq_api, model_name='llama-3.1-8b-instant')

In [ ]:
from langchain_core.documents import Document
text="""
Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space
company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,
formerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.
He is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be
US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended
the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through
his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.
Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics
 and physics. He moved to California in 1995 to attend Stanford University, but dropped out after
  two days and, with his brother Kimbal, co-founded online city guide software company Zip2.
 """
document=[Document(page_content=text)]
document

In [ ]:
!pip install --upgrade --quiet langchain-experimental

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(llm=llm)


In [ ]:
graph_documents= llm_transformer.convert_to_graph_documents(document)

In [ ]:
graph_documents

In [ ]:
graph_documents[0].nodes

In [ ]:
graph_documents[0].relationships

In [ ]:
### Load the dataset of movie

movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [ ]:
movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""
graph.query(movie_query)

In [ ]:
graph

In [ ]:
graph.query(movie_query)

In [ ]:
graph.refresh_schema()
print(graph.schema)

In [ ]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(llm=llm,graph=graph,verbose=True, allow_dangerous_requests=True)
chain

In [ ]:
response=chain.invoke({"query":"Who was the director of the moview GoldenEye"})

response


In [ ]:
response=chain.invoke({"query":"tell me count of movies of every director"})

response

In [ ]:
response=chain.invoke({"query":"Who was the director in movie Casino"})

response

In [ ]:
examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "List all the genres of the movie Schindler's List",
        "query": "MATCH (m:Movie {title: 'Schindler\'s List'})-[:IN_GENRE]->(g:Genre) RETURN g.name",
    },
    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
    {
        "question": "Which directors have made movies with at least three different actors named 'John'?",
        "query": "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'John' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name",
    },
    {
        "question": "Identify movies where directors also played a role in the film.",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name",
    },
    {
        "question": "Find the actor with the highest number of movies in the database.",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1",
    },
]

In [ ]:
import json

for example in examples:
    print(f"Question: {example['question']}")
    # Directly use the correct Cypher query from the example
    cypher_query = example['query']
    print(f"Generated Cypher: {cypher_query}") # Print the used Cypher for clarity

    try:
        # Execute the correct Cypher query against the graph
        context = graph.query(cypher_query)
        # Convert context to a string format suitable for the qa_chain
        context_str = json.dumps(context) if context else "[]"

        # Use the qa_chain to generate a natural language answer from the context
        response = chain.qa_chain.invoke({"question": example['question'], "context": context_str})
        print(f"Full Context: {context_str}") # Print context for debugging/verification
        print(f"Answer: {response['text']}") # Directly print the 'text' field of the response
    except Exception as e:
        print(f"Error executing Cypher query: {e}")
        print(f"Answer: I don't know the answer due to an error.")
    print("---\n")

In [ ]:
for example in examples:
    print(f"Question: {example['question']}")
    # Directly use the correct Cypher query from the example
    cypher_query = example['query']
    print(f"Generated Cypher: {cypher_query}") # Print the used Cypher for clarity

    try:
        # Execute the correct Cypher query against the graph
        context = graph.query(cypher_query)
        # Convert context to a string format suitable for the qa_chain
        context_str = json.dumps(context) if context else "[]"

        # Use the qa_chain to generate a natural language answer from the context
        response = chain.qa_chain.invoke({"question": example['question'], "context": context_str})
        print(f"Full Context: {context_str}") # Print context for debugging/verification
        print(f"Answer: {response['text']}") # Directly print the 'text' field of the response
    except Exception as e:
        print(f"Error executing Cypher query: {e}")
        print(f"Answer: I don't know the answer due to an error.")
    print("---\n")